# Notebook 02: Catalog API Basics

**Objective:** Learn CRUD operations on catalog metadata via the OpenMetadata REST API.

**Prerequisites:**
- Completed Notebook 01 (catalog lab running, ecommerce_postgres service registered)
- Metadata ingestion has been run (4 tables discovered: customers, products, orders, order_items)

**What you will learn:**
1. Get table details by Fully Qualified Name (FQN)
2. Add descriptions to tables via PATCH requests
3. Add column-level descriptions
4. Update table owner
5. View change history (activity feed)

## Setup: Configuration and Authentication

In [ ]:
import requests
import json
import time

# OpenMetadata API base URL (internal Docker network)
OM_URL = "http://openmetadata-server:8585/api/v1"

def get_auth_token(base_url, username="admin", password="admin", max_retries=12, wait_seconds=15):
    """Authenticate with OpenMetadata and return JWT token."""
    login_url = f"{base_url}/users/login"
    payload = {"email": username, "password": password}
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(login_url, json=payload, timeout=10)
            if resp.status_code == 200:
                print(f"Authenticated successfully")
                return resp.json()["accessToken"]
            print(f"Attempt {attempt}/{max_retries}: HTTP {resp.status_code} - retrying...")
        except (requests.ConnectionError, requests.Timeout):
            print(f"Attempt {attempt}/{max_retries}: Server not ready - retrying...")
        time.sleep(wait_seconds)
    raise ConnectionError("Could not connect to OpenMetadata")

token = get_auth_token(OM_URL)
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# PATCH requests use JSON Patch format (RFC 6902)
patch_headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json-patch+json"}

print("Ready for API exercises.")

## Exercise 1: Get Table Details by FQN

Every entity in OpenMetadata has a **Fully Qualified Name (FQN)** that uniquely identifies it:

```
<service>.<database>.<schema>.<table>
```

For our lab: `ecommerce_postgres.ecommerce.public.<table_name>`

In [ ]:
# Define FQNs for all our e-commerce tables
SERVICE = "ecommerce_postgres"
DATABASE = "ecommerce"
SCHEMA = "public"
FQN_PREFIX = f"{SERVICE}.{DATABASE}.{SCHEMA}"

TABLE_NAMES = ["customers", "products", "orders", "order_items"]

# Fetch details for each table
for table_name in TABLE_NAMES:
    fqn = f"{FQN_PREFIX}.{table_name}"
    resp = requests.get(
        f"{OM_URL}/tables/name/{fqn}",
        headers=headers,
        params={"fields": "columns,owner,tags"}
    )
    
    if resp.status_code == 200:
        table = resp.json()
        cols = table.get("columns", [])
        owner = table.get("owner", {}).get("name", "No owner")
        desc = table.get("description", "No description")
        print(f"Table: {fqn}")
        print(f"  Description: {desc[:80]}")
        print(f"  Owner: {owner}")
        print(f"  Columns: {', '.join(c['name'] for c in cols)}")
        print()
    else:
        print(f"Table not found: {fqn} (HTTP {resp.status_code})")
        print(f"  Make sure metadata ingestion has been run. See Notebook 01, Exercise 3.")
        print()

## Exercise 2: Add Descriptions to Tables

Good catalog practice: every table should have a clear description explaining its purpose. We use the **PATCH** method with JSON Patch format (RFC 6902) to add descriptions without overwriting other metadata.

**Why PATCH, not PUT?** PUT replaces the entire entity. PATCH modifies specific fields, which is safer for concurrent catalog operations.

In [ ]:
# Table descriptions to add
table_descriptions = {
    "customers": "Core customer entity containing personal information (PII), contact details, and account timestamps. Source of truth for customer identity across the e-commerce platform.",
    "products": "Product catalog with pricing and inventory data. Categories: electronics, clothing, books, home, sports. Prices are in USD.",
    "orders": "Customer purchase orders tracking the full lifecycle from pending to delivered/cancelled. Links to customers via customer_id FK.",
    "order_items": "Line-item detail for each order, linking orders to specific products with quantity and unit price at time of purchase."
}

for table_name, description in table_descriptions.items():
    fqn = f"{FQN_PREFIX}.{table_name}"
    
    # JSON Patch: add or replace the description field
    patch_payload = [
        {"op": "add", "path": "/description", "value": description}
    ]
    
    resp = requests.patch(
        f"{OM_URL}/tables/name/{fqn}",
        headers=patch_headers,
        json=patch_payload
    )
    
    if resp.status_code == 200:
        print(f"Updated description for '{table_name}'")
    else:
        print(f"Failed to update '{table_name}': HTTP {resp.status_code} - {resp.text[:100]}")

print("\nAll table descriptions updated.")

## Exercise 3: Add Column-Level Descriptions

Column descriptions are critical for data understanding. Let's add descriptions to key columns in the `customers` table.

**Why column descriptions matter:** Without them, consumers must guess what `updated_at` means (last profile update? last order? last login?). Column descriptions eliminate ambiguity.

In [ ]:
# First, get the current table to find column indices
fqn = f"{FQN_PREFIX}.customers"
resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers, params={"fields": "columns"})
table = resp.json()
columns = table.get("columns", [])

# Column descriptions to add
column_descriptions = {
    "email": "Unique email address. Used for login, notifications, and as a natural key. Contains PII.",
    "first_name": "Customer's first (given) name. Contains PII.",
    "last_name": "Customer's last (family) name. Contains PII.",
    "phone": "Contact phone number in international format (+7-XXX-XXX-XXXX). Optional field. Contains PII.",
    "created_at": "Timestamp of initial account registration. Immutable after creation.",
    "updated_at": "Timestamp of last profile modification (name, email, phone changes). Does NOT reflect order activity."
}

# Build JSON Patch operations for each column
patch_ops = []
for i, col in enumerate(columns):
    col_name = col.get("name", "")
    if col_name in column_descriptions:
        patch_ops.append({
            "op": "add",
            "path": f"/columns/{i}/description",
            "value": column_descriptions[col_name]
        })

# Apply all column description patches at once
resp = requests.patch(f"{OM_URL}/tables/name/{fqn}", headers=patch_headers, json=patch_ops)

if resp.status_code == 200:
    updated = resp.json()
    print(f"Updated {len(patch_ops)} column descriptions for 'customers':")
    for col in updated.get("columns", []):
        desc = col.get("description", "-")
        print(f"  {col['name']}: {desc[:60]}{'...' if len(desc) > 60 else ''}")
else:
    print(f"Failed: HTTP {resp.status_code} - {resp.text[:200]}")

## Exercise 4: Update Table Owner

Ownership is a governance fundamental. Every table should have a clear owner responsible for its quality, documentation, and access policies.

Let's assign the `admin` user as owner of the customers table.

In [ ]:
# First, find the admin user's ID
resp = requests.get(f"{OM_URL}/users/name/admin", headers=headers)
if resp.status_code == 200:
    admin_user = resp.json()
    admin_id = admin_user["id"]
    print(f"Admin user ID: {admin_id}")
else:
    print(f"Could not find admin user: {resp.status_code}")
    admin_id = None

if admin_id:
    # Assign owner to each table
    for table_name in TABLE_NAMES:
        fqn = f"{FQN_PREFIX}.{table_name}"
        patch_payload = [
            {
                "op": "add",
                "path": "/owner",
                "value": {
                    "id": admin_id,
                    "type": "user"
                }
            }
        ]
        resp = requests.patch(f"{OM_URL}/tables/name/{fqn}", headers=patch_headers, json=patch_payload)
        if resp.status_code == 200:
            owner = resp.json().get("owner", {}).get("name", "unknown")
            print(f"  {table_name}: owner set to '{owner}'")
        else:
            print(f"  {table_name}: failed ({resp.status_code})")
    
    print("\nAll tables now have an owner assigned.")

## Exercise 5: View Change History

OpenMetadata tracks all metadata changes. Let's view the activity feed to see the changes we just made (descriptions, ownership).

This is valuable for **audit trails** -- knowing who changed what metadata and when.

In [ ]:
# Get the change events (activity feed)
# The events API shows recent metadata changes across the catalog
resp = requests.get(
    f"{OM_URL}/feed",
    headers=headers,
    params={"type": "Conversation", "filterType": "OWNER", "limit": 20}
)

if resp.status_code == 200:
    feed = resp.json().get("data", [])
    print(f"Recent activity feed entries: {len(feed)}")
    print("=" * 70)
    for entry in feed[:10]:  # Show latest 10
        about = entry.get("about", "")
        msg = entry.get("message", "No message")
        created = entry.get("updatedAt", "")
        print(f"  About: {about[:60]}")
        print(f"  Message: {msg[:80]}")
        print(f"  Time: {created}")
        print()
else:
    print(f"Feed request returned: {resp.status_code}")
    # Try alternative: change events endpoint
    print("\nTrying change events endpoint...")
    resp2 = requests.get(
        f"{OM_URL}/events/subscriptions",
        headers=headers
    )
    print(f"Events subscriptions: {resp2.status_code}")

In [ ]:
# Verify our changes by re-fetching the customers table
fqn = f"{FQN_PREFIX}.customers"
resp = requests.get(
    f"{OM_URL}/tables/name/{fqn}",
    headers=headers,
    params={"fields": "columns,owner,tags"}
)

if resp.status_code == 200:
    table = resp.json()
    print(f"Verification: {table['fullyQualifiedName']}")
    print(f"  Description: {table.get('description', 'None')[:80]}...")
    print(f"  Owner: {table.get('owner', {}).get('name', 'None')}")
    print(f"  Columns with descriptions:")
    for col in table.get("columns", []):
        has_desc = "YES" if col.get("description") else "NO"
        print(f"    {col['name']}: description={has_desc}")

## Summary

In this notebook, you learned:

1. **FQN pattern:** `<service>.<database>.<schema>.<table>` uniquely identifies every table
2. **PATCH operations:** Use JSON Patch (RFC 6902) to modify metadata without replacing entire entities
3. **Table descriptions:** Critical documentation that explains purpose and context
4. **Column descriptions:** Eliminate ambiguity about what each field means
5. **Ownership:** Governance requires clear accountability for every data asset
6. **Audit trail:** All changes are tracked for compliance and governance

**Key concept:** A catalog is not just for discovery -- it's a living documentation system. Regular description updates and ownership assignments keep the catalog valuable.

**Next:** [03-tagging-classification.ipynb](03-tagging-classification.ipynb) - Add tags, classify PII columns, and apply governance labels.